# 🔴 Solution: LoRA Linear Layer

### Exercise contract
Expose public tensors `self.W` `(out_features, in_features)`, `self.A` `(rank, in_features)`, and `self.B` `(out_features, rank)`, plus numeric attributes `self.rank` and `self.alpha`. Register A and B as trainable `nn.Parameter`s when `rank > 0`; the base W may be frozen or trainable. Tests set A/B to nonzero values to check the update.

This exercise is bias-free. Input `(..., in_features)` maps to `(..., out_features)` and must support autograd. Support `rank=0` by returning `x @ W.T` without evaluating `alpha / rank`; A/B may have zero-sized rank dimensions.

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class LoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int, alpha: float = 1.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        
        # Base weight
        self.W = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.W)
        
        if rank > 0:
            # LoRA matrices
            self.A = nn.Parameter(torch.empty(rank, in_features))
            self.B = nn.Parameter(torch.zeros(out_features, rank))
            nn.init.kaiming_uniform_(self.A)
        else:
            self.A = nn.Parameter(torch.empty(0, in_features), requires_grad=False)
            self.B = nn.Parameter(torch.empty(out_features, 0), requires_grad=False)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base output
        out = x @ self.W.T
        
        # LoRA addition
        if self.rank > 0:
            out = out + (x @ self.A.T @ self.B.T) * (self.alpha / self.rank)
        
        return out

In [ ]:
# Verify
lora = LoRALinear(in_features=64, out_features=128, rank=8)
x = torch.randn(2, 10, 64)
out = lora(x)
print(f"Output shape: {out.shape}")
print(f"B sum (should be ~0): {lora.B.sum().item()}")

In [ ]:
from torch_judge import check
check("lora")